Import Dependencies

In [1]:
import openai
import pandas as pd
import cohere

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

Retrieval

In [3]:
query = "Can I get a tablet?"

In [4]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [5]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [6]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [7]:
results = retrieve_data(query, k=20)

In [8]:
results

{'retrieved_context_ids': ['B0B7QL4FMN',
  'B0C4KCQB19',
  'B09MKYBJT6',
  'B07FMF2L6B',
  'B0BN7JTB5N',
  'B0B3QGL9C9',
  'B0BCFYCXRH',
  'B0BCDYMF5L',
  'B0C1TWNBGD',
  'B0BQ1ZVF2C',
  'B0BHP1HMK2',
  'B0BJPTYFK9',
  'B0BKZN3N6L',
  'B0B7MR7WS7',
  'B0BDD6BC6Y',
  'B0C1P9YW5R',
  'B0BTPG4XHH',
  'B09RN3KN5C',
  'B0C6QXGW5S',
  'B09SFX1KYC'],
 'retrieved_context': ['GrandPad Senior Tablet (Renewed) with Phone Capabilities, 4G LTE, Wireless Charger, Stylus, 1 Month Premium Service Plan Included, Purchase A Plan at Activation MADE FOR SENIORS: The GrandPad is the ideal tablet & phone, all-in-one connectivity device for older adults. It has a closed network, so it keeps the older adult free of receiving any spam or robocalls. Its simple interface & easy connectivity makes it the perfect choice for families looking to stay connected in a safe & secure way. SERVICE PLAN NEEDED: 1 Month Included - A Monthly or Annual Service Plan can be purchased at Activation by calling GrandPad directly. 

Reranking

In [9]:
cohere_client = cohere.ClientV2()
to_rerank = results["retrieved_context"]

In [10]:
to_rerank
query

'Can I get a tablet?'

In [11]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20
)

In [12]:
response

V2RerankResponse(id='4d615c5c-bb75-4ffe-9de6-f7a59b8ce8c9', results=[V2RerankResponseResultsItem(index=17, relevance_score=0.8632072), V2RerankResponseResultsItem(index=13, relevance_score=0.84826964), V2RerankResponseResultsItem(index=10, relevance_score=0.84726137), V2RerankResponseResultsItem(index=5, relevance_score=0.8442036), V2RerankResponseResultsItem(index=7, relevance_score=0.8379385), V2RerankResponseResultsItem(index=6, relevance_score=0.8374073), V2RerankResponseResultsItem(index=12, relevance_score=0.8358054), V2RerankResponseResultsItem(index=19, relevance_score=0.82592696), V2RerankResponseResultsItem(index=8, relevance_score=0.8253646), V2RerankResponseResultsItem(index=2, relevance_score=0.8236691), V2RerankResponseResultsItem(index=1, relevance_score=0.8225315), V2RerankResponseResultsItem(index=0, relevance_score=0.8196625), V2RerankResponseResultsItem(index=3, relevance_score=0.7546258), V2RerankResponseResultsItem(index=15, relevance_score=0.73685527), V2RerankRes

In [13]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [14]:
reranked_results

['Pad ',
 "TECLAST Tablet 10 inch Android 12 Tablets, P25T 3GB RAM+64GB ROM(TF 1TB), Wi-Fi 6 Tablet PC, 1.8Ghz 4-core CPU, 2.4G/5G WiFi, 800x1280 FHD IPS, Bluetooth 5.0, Dual Cameras, 6000 mAh, Stereo 【Latest Android 12 tablet + GMS certification】This latest Android 12 smart operating system in 2022 simplifies system interaction design, optimizes the underlying code of the system, and makes Android 12 P25T tablet run smoother and faster. Further security and privacy improvements for Android 12 tablets give users a deeper understanding and control over the data applications access. The 10 inch tablet PC has passed Google's GMS certification. You can not only experience stable and smooth Google services, but also install your favorite third-party applications such as LINE/Netflix/YouTube/Facebook/Twitter. 【Wi-Fi 6.0+Bluetooth 5.0+ 5000mAh】10 inch tablet support low interference 802.11A /b/g/n/ac/ax Wi-Fi 6. It has a more stable Wifi wireless signal, achieves faster transmission speed, an